# Curvature Sensitivity Analysis

**Paper:** Section 7.6  

Reproduces the curvature sensitivity sweep:
- GeoNet-fixed-c with c ∈ {-0.1, -0.5, -1.0, -2.0, -5.0} on WordNet-Mammals
- Learnable curvature convergence values (GeoNet-LC)
- Shows MAP peaks at c = -1.0 for WordNet, c ≈ -0.5 for ogbn-arxiv

**Prerequisite:** Run `scripts/reproduce/run_wordnet.sh` with curvature sweep variants.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_ROOT = Path('outputs')
SEEDS       = list(range(10))

# Curvature sweep values (Section 7.6)
C_VALUES    = [-0.1, -0.5, -1.0, -2.0, -5.0]
DATASETS    = ['wordnet_mammals', 'ogbn_arxiv']
METRICS     = {'wordnet_mammals': 'MAP', 'ogbn_arxiv': 'accuracy'}

def load_curvature_results(dataset, c_val, seeds=SEEDS):
    """Load results for GeoNet-fixed-c with specific curvature."""
    c_str = str(c_val).replace('-', 'neg').replace('.', 'p')
    vals  = []
    for seed in seeds:
        p = OUTPUT_ROOT / f'geonet_fixed_c{c_str}_{dataset}_seed{seed}' / 'results.json'
        if p.exists():
            metric = METRICS[dataset]
            with open(p) as f:
                r = json.load(f)
            vals.append(r['test_metrics'].get(metric, np.nan))
    return np.array(vals) if vals else np.array([np.nan] * len(seeds))

# Also load learnable curvature values (GeoNet-LC, learn_c=True)
def load_learned_curvature(dataset, seeds=SEEDS):
    """Load the learned curvature value at convergence."""
    learned_c = []
    for seed in seeds:
        p = OUTPUT_ROOT / f'geonet_{dataset}_seed{seed}' / 'results.json'
        if p.exists():
            with open(p) as f:
                r = json.load(f)
            learned_c.append(r.get('learned_curvature', np.nan))
    return np.array(learned_c)

print('Loading curvature sensitivity results...')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, dataset in zip(axes, DATASETS):
    metric = METRICS[dataset]
    means, stds = [], []

    for c in C_VALUES:
        vals = load_curvature_results(dataset, c)
        means.append(np.nanmean(vals))
        stds.append(np.nanstd(vals))

    means = np.array(means)
    stds  = np.array(stds)
    x     = np.arange(len(C_VALUES))

    ax.errorbar(x, means, yerr=stds, marker='o', color='#2E618A',
                linewidth=2, markersize=7, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in C_VALUES])
    ax.set_xlabel('Fixed Curvature c', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'{dataset} — Curvature Sensitivity', fontsize=12)
    ax.grid(True, alpha=0.3)

    # Mark the optimal
    best_idx = np.nanargmax(means)
    ax.axvline(best_idx, color='red', linestyle='--', alpha=0.5,
               label=f'Best: c={C_VALUES[best_idx]}')
    ax.legend()

plt.tight_layout()
import os; os.makedirs('outputs/tables', exist_ok=True)
plt.savefig('outputs/tables/curvature_sensitivity.pdf', bbox_inches='tight')
plt.show()
print('Figure saved to outputs/tables/curvature_sensitivity.pdf')

In [ ]:
# Learned curvature convergence values
print('\n=== Learned Curvature at Convergence (GeoNet-LC) ===')
for dataset in DATASETS:
    c_learned = load_learned_curvature(dataset)
    if not np.all(np.isnan(c_learned)):
        print(f'  {dataset}: c = {np.nanmean(c_learned):.3f} ± {np.nanstd(c_learned):.3f}')
    else:
        print(f'  {dataset}: learned curvature data not found in results.json')
        print('  (Add "learned_curvature" key to results in train.py to log this.)')

print('\nSection 7.6 analysis complete.')